# Análisis de Datos:

## Introducción
### Definición de Variedad
Formalmente, una variedad $X$, de dimensión $n$ (n-variedad), es un espacio topológico de Hausdorff, 2-numerable tal que $\forall x \in X$ existe una vecindad $V$ homeomorfa a un abierto $U$ en el plano $\mathbb{R}^n$.

### Ejemplos de variedades
- Una línea, un plano, una ésfera.
- Una 2-variedad es una superficie; si es compacta y no tiene frontera, se llama superficie cerrada.
- El conjunto $G_k(\mathbb{R}^n) = \{L \subset \mathbb{R}^n\ |\ dim_{\mathbb{R}}\ L = k\}$ de espacios lineales de dimensión $k$ conocido como variedad de Grassmann.

### Hipotésis de la variedad
Postula que los puntos de un conjunto de datos yacen en una variedad (subvariedad o subespacio) de dimensión menor que el espacio en el que están inmersas.

### Aprendizaje en variedades
El aprendizaje de variedades es una clase de algoritmos que intenta recuperar una variedad de dimensión baja inmersa en un espacio ambiente de dimensión alta.

![manifold](./manifold_unfolding.png)

### Reducción de dimensión
Es una familia de algoritmos que intenta hallar una representación de los datos de menor dimensión que, a la vez, preserve las características clave del conjunto de datos.

#### Reducción de Dimensión en el Proceso de Análisis de Datos
Los algoritmos de reducción de dimensión son una herramienta útil en:
- El análisis exploratorio
- El pre-procesamiento de datos
- La visualización de datos 


### Análisis de Componentes Principales
Es el algoritmo más popular de reducción de dimensión.
Existen una gran cantidad de variantes: PCA supervisado (SPCA), PCA probabilístico (PPCA), PCA dual y kernel PCA.

El algoritmo original asume que los datos datos yacen en una varidedad lineal y por medio de una proyección lineal, "desenrrolla" la variedad en un espacio que maximiza la varianza de los datos.

Formalmente, se puede expresar como el siguiente problema de optimización:

$$
\begin{align*} 
\underset{U}{\text{max }}\  tr(U^\intercal S U)\\
\text{sujeto a }\  U^\intercal U = I
\end{align*}
$$

Que se reduce al problema de valores propios

$$
S U = U \Lambda
$$

Para reducir la dimensión de los datos hay que decidir el número de vectores propios a retener, 
generalmente siguiendo la siguiente regla:

$$
\sum_{j = 1}^p \frac{\lambda_j}{\sum_{k = 1}^d \lambda_k} \geq 0.8
$$

Que se interpreta como conservar los $p$ vectores propios que llevan el menos el $80%$ de 
varianza.

### Espacios de Hilbert con Kernel Reproducible

### Ejemplo 1: Datos Círculares

La librería `sklearn` de Python provee la función `make_circles` que genera datos círculares. Es ideal para mostrar un conjunto de datos que PCA no puede manejar
pero sí KPCA.
La función regresa el conjunto de datos `X` y una matriz `y` que contiene información sobre el conglomerado al que pertenece cada punto.

In [6]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.datasets import make_circles
from sklearn.decomposition import KernelPCA
from sklearn.metrics.pairwise import pairwise_kernels

# Generar Conjunto de Datos
X, y = make_circles(n_samples=250, factor=0.2, noise=0.05, random_state=0)

kpca_linear = KernelPCA(n_components=2, kernel='linear', alpha = 0.1, gamma=10, fit_inverse_transform=False)
kpca_rbf = KernelPCA(n_components=2, kernel='rbf', alpha = 0.1, gamma=10, fit_inverse_transform=False)
kpca_cosine = KernelPCA(n_components=2, kernel='cosine', alpha = 0.1, gamma=10, fit_inverse_transform=False)

X_linear = kpca_linear.fit_transform(X)
X_rbf = kpca_rbf.fit_transform(X)
X_cosine = kpca_cosine.fit_transform(X)

# Convertir a Dataframe
X = pd.DataFrame(X)
y = pd.DataFrame(y)
X_linear = pd.DataFrame(X_linear)
X_rbf = pd.DataFrame(X_rbf)
X_cosine = pd.DataFrame(X_cosine)

# Agrega columna de etiquetas
X['class'] = y

In [7]:
# Calcular Matriz Kernel de los datos y de los datos ordenados
kernel_matrix = pairwise_kernels(X, metric=kpca_rbf.kernel, gamma=kpca_rbf.gamma)
kernel_matrix_sorted = pairwise_kernels(X.sort_values(by='class'), metric=kpca_rbf.kernel, gamma=kpca_rbf.gamma)

In [39]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

kernel_matrices = make_subplots(rows=1, cols=2, subplot_titles=("Unsorted", "Sorted"))

kernel_matrices.add_trace(
    go.Heatmap(
        z=kernel_matrix,
        colorscale="Magma",
        colorbar=dict(x=0.45)
    ),
    row=1, col=1
)

kernel_matrices.add_trace(
    go.Heatmap(
        z=kernel_matrix_sorted,
        colorscale="Magma",
        colorbar=dict(x=0.45)
    ),
    row=1, col=2
)

kernel_matrices.update_yaxes(autorange="reversed")
kernel_matrices.show()


![Kernel Matrix](../img/kernel_matrix_one.png)

Lo anterior corresponde a una visualización de las matrices kernel para el kernel RBF.
Se distinguen dos conjuntos de datos similares entre ellos, pero no entre conjuntos de datos.
Si construimos una gráfica conectado dos puntos sii $K(i,j) \neq 0$, entonces la gráfica tendría dos conjuntos conexos.

In [38]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

marker_style = dict(color=X['class'], colorscale="rdbu")

fig = make_subplots(rows=2, cols=2, subplot_titles=("Original", "Linear", "RBF", "Cosine"))

fig.add_trace(go.Scatter(x=X[0], y=X[1], mode='markers', name='Original', marker=marker_style), row=1, col=1)
fig.add_trace(go.Scatter(x=X_linear[0], y=X_linear[1], mode='markers', name='Linear', marker=marker_style), row=1, col=2)
fig.add_trace(go.Scatter(x=X_rbf[0], y=X_rbf[1], mode='markers', name='RBF', marker=marker_style), row=2, col=1)
fig.add_trace(go.Scatter(x=X_cosine[0], y=X_cosine[1], mode='markers', name='Cosine', marker=marker_style), row=2, col=2)

fig.update_layout(showlegend=False)
fig.show()


![Comparison Of kernels](../img/circular_kernel_comparison.png)

Podemos dar la siguiente interpretación:
- El kernel lineal solo rota los datos en la dirección de máxima varianza, i.e, el RKHS generado por el kernel lineal es igual al original pero rotado en la
  dirección de máxima varianza.
- Los datos en el RKHS generado por el kernel RBF caen en una variedad lineal de modo que al aplicar PCA se forman dos conglomerados.
- Los datos en el RKHS generado por el kernel Coseno caen en una variedad no-lineal, al aplicar PCA los datos se distribuyen en torno a una elipse. 